# Python Source Map: Ollama-Assisted Alternative-Proposal Detection & Timing

For each decision episode (`Episode #`), this notebook identifies comments that introduce a **new, distinct alternative proposal**, excluding the original proposal.

This version is completely local/free after installing Ollama and downloading an open-source model. It uses:

1. A local Ollama instruction-tuned model to classify comments in context.
2. Structured JSON output: `new_alternative`, `confidence`, and a short rationale.
3. Sequential episode processing so previously identified alternatives can be supplied as context and duplicate proposals are not counted repeatedly.
4. Episode-level aggregation into alternative counts and latest-alternative timing.
5. Validation against the supplied `Discussion | Alternatives | Alternative Timing` baseline.

**Important methodological limitation:** the supplied baseline contains episode-level counts, not comment-level gold labels. Validation therefore tests agreement with the existing episode-level coding; it is not a comment-level precision/recall/F1 evaluation.


## Cell 1: Configuration, MongoDB connection, and Ollama setup

This cell connects to the existing MongoDB corpus and a **local Ollama server**. No paid API or API key is required.

Before running the notebook:

1. Install Ollama from https://ollama.com/download
2. Start Ollama.
3. In Terminal, run `ollama pull qwen3:4b` (or change `OLLAMA_MODEL` below).
4. Keep Ollama running while the notebook executes.


In [30]:
import json
import os
import re
import math
import random
import json as jsonlib
import numpy as np
import pandas as pd
import requests
from pymongo import MongoClient

with open("config.json") as f:
    config = json.load(f)

mongo_uri = config["mongo_uri"]

client = MongoClient(mongo_uri)
db = client["consensus"]
source_map_coll = db["python_source_map"]
comments_coll = db["python_source_map_comments"]

# Ollama configuration ------------------------------------------------------
OLLAMA_URL = config.get("ollama_url", "http://localhost:11434")
OLLAMA_MODEL = "qwen3:1.7b"  # or "qwen3:1.7b-nolimits" depending on what you pulled
OLLAMA_TIMEOUT = int(config.get("alternative_detection_ollama_timeout", 60))
OLLAMA_TEMPERATURE = float(config.get("alternative_detection_ollama_temperature", 0.0))

EPISODE_FIELD = "Episode #"

# Check that Ollama is reachable and the requested model is installed.
try:
    tags = requests.get(f"{OLLAMA_URL}/api/tags", timeout=10).json()
    installed_models = {m.get("name", "") for m in tags.get("models", [])}
except Exception as exc:
    raise RuntimeError(
        "Could not reach Ollama at " + OLLAMA_URL + ". Start Ollama first. "
        "On macOS, open the Ollama app or run `ollama serve`."
    ) from exc

if not any(OLLAMA_MODEL == name or OLLAMA_MODEL in name for name in installed_models):
    raise RuntimeError(
        f"Ollama model {OLLAMA_MODEL!r} is not installed. Run `ollama pull {OLLAMA_MODEL}` "
        "in Terminal, or change OLLAMA_MODEL in this cell."
    )

print("Connected to MongoDB database 'consensus'.")
print(f"  - {source_map_coll.name}: {source_map_coll.estimated_document_count()} existing documents")
print(f"  - {comments_coll.name}: {comments_coll.estimated_document_count()} existing documents")
print(f"Ollama model: {OLLAMA_MODEL}")


Connected to MongoDB database 'consensus'.
  - python_source_map: 130 existing documents
  - python_source_map_comments: 4068 existing documents
Ollama model: qwen3:1.7b


In [31]:
SAMPLE_EPISODES = [1, 100, 2, 3, 4, 52, 6]

# Match both integer and string representations of episode keys
search_keys = SAMPLE_EPISODES + [str(e) for e in SAMPLE_EPISODES]
query = {EPISODE_FIELD: {"$in": search_keys}}

episode_comments = {}
for doc in comments_coll.find(query):
    ep_key = doc.get(EPISODE_FIELD)
    # Extract comment text from any common MongoDB field name
    text = doc.get("comment") or doc.get("text") or doc.get("body") or doc.get("content") or ""
    if text:
        # Standardize key as int if numeric, otherwise str
        clean_key = int(ep_key) if str(ep_key).isdigit() else ep_key
        episode_comments.setdefault(clean_key, []).append(text.strip())

print(f"Loaded comments for {len(episode_comments)} episodes:")
for ep, comments in sorted(episode_comments.items(), key=lambda x: str(x[0])):
    print(f"  - Episode {ep}: {len(comments)} comment(s)")

Loaded comments for 7 episodes:
  - Episode 1: 18 comment(s)
  - Episode 100: 13 comment(s)
  - Episode 2: 9 comment(s)
  - Episode 3: 4 comment(s)
  - Episode 4: 5 comment(s)
  - Episode 52: 88 comment(s)
  - Episode 6: 101 comment(s)


In [ ]:
import time
import requests
import re
import json

def extract_alternative_proposals(ep_id: int, comments: list[str], max_retries: int = 2) -> dict:
    """Uses Ollama to extract raw alternative proposals with retry logic and context capping."""
    if not comments:
        return {"episode_id": ep_id, "proposals": [], "rationale": "No comments found for episode."}

    # Truncate individual comments if extremely long to avoid context wall
    cleaned_comments = [c[:1000] for c in comments[:30]] 
    formatted_comments = "\n".join(f"[{i+1}] {c}" for i, c in enumerate(cleaned_comments))

    prompt = f"""You are evaluating technical discussion comments for Episode {ep_id}.

TASK:
Extract all alternative proposals (excluding the original proposal) mentioned in the comments. Follow a strict definition:The alternative proposal should be mutually exclusive with the original proposal, not something that can be incorporated into the existing proposal.

COMMENTS:
{formatted_comments}

INSTRUCTIONS:
- Extract each distinct proposal as an item in "proposals".
- Output ONLY valid JSON matching this schema:
{{
  "episode_id": {ep_id},
  "proposals": [
    {{
      "comment_index": <number>,
      "proposal_text": "<summary of alternative>",
      "type": "<'alternative' or 'counter-argument'>"
    }}
  ],
  "rationale": "<1-sentence summary>"
}}
"""

    payload = {
        "model": OLLAMA_MODEL,
        "prompt": prompt,
        "stream": False,
        "format": "json",
        "keep_alive": "10m",
        "options": {
            "temperature": 0.1,
            "num_ctx": 4096,     # Limits context window size to prevent RAM overflow
            "num_predict": 2048,  # Provides adequate room for output tokens
        }
    }

    for attempt in range(max_retries + 1):
        try:
            response = requests.post(
                f"{OLLAMA_URL}/api/generate",
                json=payload,
                timeout=OLLAMA_TIMEOUT
            )
            response.raise_for_status()
            
            raw_text = response.json().get("response", "").strip()

            # Extract outermost JSON object
            json_match = re.search(r"\{.*\}", raw_text, re.DOTALL)
            if not json_match:
                return {
                    "episode_id": ep_id, 
                    "proposals": [], 
                    "rationale": f"No JSON found in output: {raw_text[:60]}..."
                }

            output = json.loads(json_match.group(0))
            return {
                "episode_id": ep_id,
                "proposals": output.get("proposals", []),
                "rationale": str(output.get("rationale", "N/A"))
            }

        except requests.exceptions.HTTPError as err:
            # Handle 500 internal server error with retry delay
            if attempt < max_retries:
                time.sleep(2)  # Wait for Ollama to reset memory
                continue
            return {
                "episode_id": ep_id, 
                "proposals": [], 
                "rationale": f"Ollama HTTP {err.response.status_code} Error: {err}"
            }
        except Exception as err:
            return {
                "episode_id": ep_id, 
                "proposals": [], 
                "rationale": f"Error: {err}"
            }

In [38]:
import json

raw_json_results = []
total = len(SAMPLE_EPISODES)

for i, ep_id in enumerate(SAMPLE_EPISODES, start=1):
    # \r resets line, \033[K clears remaining leftover characters from previous line
    print(f"\r\033[K[{i}/{total}] Extracting proposals for Episode {ep_id}...", end="", flush=True)
    
    comments = episode_comments.get(ep_id, [])
    res = extract_alternative_proposals(ep_id, comments)
    raw_json_results.append(res)

print("\n\nExtraction completed!")

# ensure_ascii=False renders unicode characters directly (e.g. 'Tvrtković' instead of '\u0107')
raw_json_output = json.dumps(raw_json_results, indent=2, ensure_ascii=False)

print("\n=== Raw Extracted Proposals (JSON) ===")
print(raw_json_output)

[7/7] Extracting proposals for Episode 6.....

Extraction completed!

=== Raw Extracted Proposals (JSON) ===
[
  {
    "episode_id": 1,
    "proposals": [
      {
        "comment_index": 3,
        "proposal_text": "Jelle Zijlstra suggests labeling issues with PEP numbers to improve clarity and track changes.",
        "type": "alternative"
      },
      {
        "comment_index": 6,
        "proposal_text": "Jelle Zijlstra proposes reorganizing the spec into multiple documents to address gaps and improve structure.",
        "type": "alternative"
      },
      {
        "comment_index": 9,
        "proposal_text": "The user suggests grouping Literals, Enums, and Unions into a single section to clarify their relationships.",
        "type": "alternative"
      },
      {
        "comment_index": 15,
        "proposal_text": "The user proposes mentioning the type_check_only directive to address underutilized features in stubs.",
        "type": "alternative"
      },
      {
        

In [ ]:
import json
import pandas as pd
from pymongo import MongoClient

# --- Option A: Load from LLM Raw JSON Output Variable ---
# raw_json_str = llm_response.strip()
# proposals = json.loads(raw_json_str)

# --- Option B: Retrieve directly from MongoDB Collection ---
client = MongoClient("mongodb://localhost:27017/")
db = client["podcast_db"]
collection = db["alternative_proposals"]

proposals = list(collection.find({}, {"_id": 0}))

# Convert to DataFrame
df = pd.DataFrame(proposals)

# Aggregate counts and summarize proposals by episode
summary_df = df.groupby("episode_id").agg(
    total_proposals=("proposal_id", "count"),
    unique_speakers=("speaker", lambda x: list(set(x))),
    all_proposals=("proposal_text", list)
).reset_index()

# Display summary table
print("=== Summary by Episode ===")
print(summary_df[["episode_id", "total_proposals", "unique_speakers"]].to_string(index=False))

# Detailed Breakdown
print("\n=== Detailed Episode Breakdown ===")
for _, row in summary_df.iterrows():
    print(f"\n--- Episode: {row['episode_id']} ({row['total_proposals']} proposals) ---")
    for i, prop in enumerate(row['all_proposals'], 1):
        print(f"  {i}. {prop}")